**ESTUDIANTES:**


Angie Melissa Aguilar Alemán


Magdaleno Gómez Díaz


Sharon Dayana Blanco Piedra


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = '/content/drive/My Drive/Proyecto/archivo_final_proyecto_entrega.csv'

try:
    df = pd.read_csv(file_path)
    print(f"CSV '{file_path}' cargado exitosamente.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: El archivo no se encontró en la ruta: {file_path}")
    print("Por favor, verifica la ruta y el nombre del archivo en tu Google Drive.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo CSV: {e}")

CSV '/content/drive/My Drive/Proyecto/archivo_final_proyecto_entrega.csv' cargado exitosamente.


,Season,player_id,short_name,age,nationality,club_name,league_name,player_positions,club_position,preferred_foot,...,preferred_foot_valido,GrupoAltura,GrupoValor,GrupoEdad,BrechaPotencial,PosicionPrincipal,EficienciaFinanciera,flag_habilidades_bajas,flag_valor_salario,edad_sospechosa
0,2015,158023,L. Messi,27,Argentina,FC Barcelona,Spain Primera Division,CF,CF,Left,...,True,Bajo,Alto,Prime,2,CF,182.73,0,0,0
1,2015,20801,Cristiano Ronaldo,29,Portugal,Real Madrid CF,Spain Primera Division,"LW, LM",LW,Right,...,True,Medio,Alto,Veteran,0,LW,210.67,0,0,0
2,2015,9014,A. Robben,30,Netherlands,FC Bayern München,German 1. Bundesliga,"RM, LM, RW",SUB,Left,...,True,Medio,Alto,Veteran,0,RM,198.18,0,0,0
3,2015,41236,Z. Ibrahimović,32,Sweden,Paris Saint-Germain,French Ligue 1,ST,ST,Right,...,True,Alto,Alto,Veteran,0,ST,190.91,0,0,0
4,2015,167495,M. Neuer,28,Germany,FC Bayern München,German 1. Bundesliga,GK,GK,Right,...,True,Alto,Alto,Prime,0,GK,211.67,0,0,0


# Definir y justificar la granularidad en términos de preguntas de negocio que se quieren responder.

## Granularidad por Consulta

| Consulta | Tabla de Hecho | Granularidad |
|---|---|---|
| Top 10 clubes por promedio de valor de mercado por temporada | `fact_valor_club` | Un registro por **club por temporada** |
| Valor total de mercado por región y por liga | `fact_valor_region_liga` | Un registro por **región y por liga** |
| Rendimiento (Overall) medio por GrupoEdad y por posición | `fact_rendimiento_posicion_edadgrupo` | Un registro por **GrupoEdad y posición** |
| Análisis de márgenes (ValorMercado - Salario) por Club y por Liga | `fact_margen_club` | Un registro por **club y por liga** |
| Drill-down: Club → Liga → Región con promedio de ValorMercado y Salario | `fact_drilldown_mercado` | Un registro por **combinación club–liga–región** |

---


# Medidas de la tabla de hechos




### DIMENSIONES

club_name, league_name, region (construida desde league_name), Season, GrupoEdad, club_position

### MÉTRICAS

value_eur, wage_eur, overall, margen (valor mercado - salario)

#Procesos de negocio

* Top 10 clubes por promedio de valor de mercado por temporada.
* Valor total de mercado por región y por liga.
* Rendimiento (Overall) medio por edadgrupo y por posición.
* Análisis de márgenes (Margen = ValorMercado - Salario) por Club y por
Liga.
* Drill-down: por Club -> Liga -> Región, con promedio de ValorMercado
y Salario.

# Definir unidades y posibles transformaciones necesarias (p. ej., tasas por minuto, log-transformaciones si aplica).



*   value_eur: euros, sin transformación
*   wage_eur: euros, sin transformación
*   overall: escala 0-100, sin transformación
*   margen: calculado, value_eur - wage_eur (euros)



In [ ]:
df['margen'] = df['value_eur'] - df['wage_eur']
display(df.head())

,Season,player_id,short_name,age,nationality,club_name,league_name,player_positions,club_position,preferred_foot,...,GrupoAltura,GrupoValor,GrupoEdad,BrechaPotencial,PosicionPrincipal,EficienciaFinanciera,flag_habilidades_bajas,flag_valor_salario,edad_sospechosa,margen
0,2015,158023,L. Messi,27,Argentina,FC Barcelona,Spain Primera Division,CF,CF,Left,...,Bajo,Alto,Prime,2,CF,182.73,0,0,0,99950000
1,2015,20801,Cristiano Ronaldo,29,Portugal,Real Madrid CF,Spain Primera Division,"LW, LM",LW,Right,...,Medio,Alto,Veteran,0,LW,210.67,0,0,0,78625000
2,2015,9014,A. Robben,30,Netherlands,FC Bayern München,German 1. Bundesliga,"RM, LM, RW",SUB,Left,...,Medio,Alto,Veteran,0,RM,198.18,0,0,0,54225000
3,2015,41236,Z. Ibrahimović,32,Sweden,Paris Saint-Germain,French Ligue 1,ST,ST,Right,...,Alto,Alto,Veteran,0,ST,190.91,0,0,0,52225000
4,2015,167495,M. Neuer,28,Germany,FC Bayern München,German 1. Bundesliga,GK,GK,Right,...,Alto,Alto,Prime,0,GK,211.67,0,0,0,63200000


## Definición de Dimensiones, Jerarquías y Claves Foráneas (FK)

### Dimensiones definidas
Se identificaron 4 dimensiones a partir de las preguntas de negocio:
- **Dim Club**: contiene club_name, league_name y region. Es la dimensión más importante porque soporta la jerarquía **Region → Liga → Club** requerida por las consultas de drill-down.
- **Dim GrupoEdad**: contiene GrupoEdad. Categoriza a los jugadores por rango etario.
- **Dim Position**: contiene club_position. Permite analizar rendimiento por posición.
- **Dim Season**: contiene Season. Permite análisis temporales año a año.

### Jerarquías
La única jerarquía presente es en Dim Club:
**Region → League → Club**

La columna region no existía en el dataset original y fue construida mapeando cada liga a su región geográfica correspondiente.

Dim League no se crea como tabla independiente porque league_name ya forma parte de la jerarquía dentro de Dim Club (Region → League → Club).

### Claves Foráneas (FK)
La facts table se conecta a cada dimensión mediante las siguientes FK:
- `fact_valor_club_temporada` utiliza `season_id` y `club_id`
- `fact_rendimiento_edad_posicion` utiliza`grupoedad_id` y `posicion_id`
- `fact_margen_club_liga` utiliza `club_id`
- `fact_drilldown_mercado ` utiliza  `club_id`
-  `fact_valor_region_liga` utiliza   `club_id`


In [ ]:
print(df['league_name'].unique())

['Spain Primera Division' 'German 1. Bundesliga' 'French Ligue 1'
 'English Premier League' 'Italian Serie A' 'Turkish Süper Lig'
 'Russian Premier League' 'Portuguese Liga ZON SAGRES' 'Unknown'
 'Danish Superliga' 'USA Major League Soccer' 'Mexican Liga MX'
 'Australian Hyundai A-League' 'Ukrainian Premier League'
 'Holland Eredivisie' 'Spanish Segunda División' 'Korean K League 1'
 'Argentina Primera División' 'English League Championship'
 'Greek Super League' 'Saudi Abdul L. Jameel League'
 'South African Premier Division' 'Austrian Football Bundesliga'
 'Swiss Super League' 'German 2. Bundesliga' 'Chilian Campeonato Nacional'
 'Belgian Jupiler Pro League' 'Scottish Premiership'
 'Colombian Liga Postobón' 'Italian Serie B' 'Norwegian Eliteserien'
 'Polish T-Mobile Ekstraklasa' 'French Ligue 2' 'Swedish Allsvenskan'
 'Scottish Championship' 'English League One' 'English League Two'
 'Rep. Ireland Airtricity League' 'Campeonato Brasileiro Série A'
 'Campeonato Brasileiro Série B' 'Ja

In [ ]:
# Mapa de ligas a regiones
region_map = {
    # Europa
    'Spain Primera Division': 'Europa',
    'German 1. Bundesliga': 'Europa',
    'French Ligue 1': 'Europa',
    'English Premier League': 'Europa',
    'Italian Serie A': 'Europa',
    'Turkish Süper Lig': 'Europa',
    'Russian Premier League': 'Europa',
    'Portuguese Liga ZON SAGRES': 'Europa',
    'Danish Superliga': 'Europa',
    'Holland Eredivisie': 'Europa',
    'Spanish Segunda División': 'Europa',
    'English League Championship': 'Europa',
    'Greek Super League': 'Europa',
    'Austrian Football Bundesliga': 'Europa',
    'Swiss Super League': 'Europa',
    'German 2. Bundesliga': 'Europa',
    'Belgian Jupiler Pro League': 'Europa',
    'Scottish Premiership': 'Europa',
    'Italian Serie B': 'Europa',
    'Norwegian Eliteserien': 'Europa',
    'Polish T-Mobile Ekstraklasa': 'Europa',
    'French Ligue 2': 'Europa',
    'Swedish Allsvenskan': 'Europa',
    'Scottish Championship': 'Europa',
    'English League One': 'Europa',
    'English League Two': 'Europa',
    'Rep. Ireland Airtricity League': 'Europa',
    'Finnish Veikkausliiga': 'Europa',
    'Czech Republic Gambrinus Liga': 'Europa',
    'German 3. Bundesliga': 'Europa',
    'Croatian Prva HNL': 'Europa',
    'Romanian Liga I': 'Europa',
    'Hungarian Nemzeti Bajnokság I': 'Europa',
    'Cypriot First Division': 'Europa',
    'Ukrainian Premier League': 'Europa',
    'Ligue 1': 'Europa',
    'La Liga': 'Europa',
    'Premier League': 'Europa',
    'Bundesliga': 'Europa',
    'Pro League': 'Europa',
    'Serie A': 'Europa',
    'Super Lig': 'Europa',
    'Eredivisie': 'Europa',
    'Liga Portugal': 'Europa',
    'Jupiler Pro League': 'Europa',
    'Super League': 'Europa',
    '1. HNL': 'Europa',
    'La Liga 2': 'Europa',
    'Serie B': 'Europa',
    'Superliga': 'Europa',
    'Premiership': 'Europa',
    'Fortuna Liga': 'Europa',
    'Championship': 'Europa',
    'Ligue 2': 'Europa',
    'Allsvenskan': 'Europa',
    '2. Bundesliga': 'Europa',
    'Eliteserien': 'Europa',
    'Ekstraklasa': 'Europa',
    'League One': 'Europa',
    '1. Division': 'Europa',
    '3. Liga': 'Europa',
    'Veikkausliiga': 'Europa',
    'League Two': 'Europa',
    'National League': 'Europa',
    'English National League': 'Europa',
    'Paris Saint-Germain': 'Europa',

    # América del Norte
    'USA Major League Soccer': 'America del Norte',
    'Mexican Liga MX': 'America del Norte',
    'Major League Soccer': 'America del Norte',
    'Liga MX': 'America del Norte',

    # América del Sur
    'Argentina Primera División': 'America del Sur',
    'Chilian Campeonato Nacional': 'America del Sur',
    'Colombian Liga Postobón': 'America del Sur',
    'Campeonato Brasileiro Série A': 'America del Sur',
    'Campeonato Brasileiro Série B': 'America del Sur',
    'Uruguayan Primera División': 'America del Sur',
    'Paraguayan Primera División': 'America del Sur',
    'Ecuadorian Serie A': 'America del Sur',
    'Peruvian Primera División': 'America del Sur',
    'Liga de Fútbol Profesional Boliviano': 'America del Sur',
    'Venezuelan Primera División': 'America del Sur',
    'Liga Profesional': 'America del Sur',
    'Primera Division': 'America del Sur',
    'Primera División': 'America del Sur',
    'Liga Pro': 'America del Sur',
    'Liga BetPlay': 'America del Sur',
    'Liga 1': 'America del Sur',
    'Liga De Futbol Prof': 'America del Sur',

    # Asia
    'Korean K League 1': 'Asia',
    'Japanese J. League Division 1': 'Asia',
    'Chinese Super League': 'Asia',
    'Indian Super League': 'Asia',
    'Saudi Abdul L. Jameel League': 'Asia',
    'UAE Arabian Gulf League': 'Asia',
    'K League 1': 'Asia',
    'J-League': 'Asia',

    # Oceanía
    'Australian Hyundai A-League': 'Oceania',
    'A-League': 'Oceania',

    # África
    'South African Premier Division': 'Africa',
    'Premier Division': 'Africa',

    # Desconocido
    'Unknown': 'Desconocido',
    'Rest of World': 'Desconocido',
    'NB I.': 'Europa',  # Liga húngara
}

# Crear la columna region en el dataframe
df['region'] = df['league_name'].map(region_map)

# Verificar que no quedaron ligas sin mapear
sin_region = df[df['region'].isna()]['league_name'].unique()
print("Ligas sin región asignada:", sin_region)

Ligas sin región asignada: []


In [ ]:
# Atributos de cada dimensión
club_dimension_attributes = ['club_name', 'league_name', 'region']  # jerarquía: region → league → club
GrupoEdad_dimension_attributes = ['GrupoEdad']
position_dimension_attributes = ['club_position']
season_dimension_attributes = ['Season']

print("\nClub Dimension Attributes:")
print(club_dimension_attributes)
print("\nGrupoEdad Dimension Attributes:")
print(GrupoEdad_dimension_attributes)
print("\nPosition Dimension Attributes:")
print(position_dimension_attributes)
print("\nSeason Dimension Attributes:")
print(season_dimension_attributes)


Club Dimension Attributes:
['club_name', 'league_name', 'region']

GrupoEdad Dimension Attributes:
['GrupoEdad']

Position Dimension Attributes:
['club_position']

Season Dimension Attributes:
['Season']


In [ ]:
# Dim Club — incluye jerarquía region → league_name → club_name
dt_club_dimension = df[club_dimension_attributes].drop_duplicates().copy()
dt_club_dimension.reset_index(drop=True, inplace=True)
dt_club_dimension['club_id'] = dt_club_dimension.index + 1
print("Dim Club:", dt_club_dimension.shape)
dt_club_dimension.head()

Dim Club: (5538, 4)


,club_name,league_name,region,club_id
0,FC Barcelona,Spain Primera Division,Europa,1
1,Real Madrid CF,Spain Primera Division,Europa,2
2,FC Bayern München,German 1. Bundesliga,Europa,3
3,Paris Saint-Germain,French Ligue 1,Europa,4
4,Manchester United,English Premier League,Europa,5


In [ ]:
# Dim GrupoEdad
dt_grupoEdad_dimension = df[GrupoEdad_dimension_attributes].drop_duplicates().copy()
dt_grupoEdad_dimension.reset_index(drop=True, inplace=True)
dt_grupoEdad_dimension['grupoEdad_id'] = dt_grupoEdad_dimension.index + 1
print("Dim GrupoEdad:", dt_grupoEdad_dimension.shape)
dt_grupoEdad_dimension.head()

Dim GrupoEdad: (3, 2)


,GrupoEdad,grupoEdad_id
0,Prime,1
1,Veteran,2
2,Young,3


In [ ]:
# Dim Position
dt_position_dimension = df[position_dimension_attributes].drop_duplicates().copy()
dt_position_dimension.reset_index(drop=True, inplace=True)
dt_position_dimension['position_id'] = dt_position_dimension.index + 1
print("Dim Position:", dt_position_dimension.shape)
dt_position_dimension.head()

Dim Position: (30, 2)


,club_position,position_id
0,CF,1
1,LW,2
2,SUB,3
3,ST,4
4,GK,5


In [ ]:
# Dim Season
dt_season_dimension = df[season_dimension_attributes].drop_duplicates().copy()
dt_season_dimension.reset_index(drop=True, inplace=True)
dt_season_dimension['season_id'] = dt_season_dimension.index + 1
print("Dim Season:", dt_season_dimension.shape)
dt_season_dimension.head()

Dim Season: (10, 2)


,Season,season_id
0,2015,1
1,2016,2
2,2017,3
3,2018,4
4,2019,5


## ============================================================
## MODELO DIMENSIONAL VISUALIZACIÓN Y DEFINICIÓN DE RELACIONES
## ============================================================

## DIMENSIONES

### dim_season
| Campo | Tipo | Descripción |
|------|------|------------|
| id | PK | Identificador de la temporada |
| season | Atributo | Año o temporada |

---

### dim_club
| Campo | Tipo | Descripción |
|------|------|------------|
| id | PK | Identificador del club |
| club_name | Atributo | Nombre del club |
| league_name | Atributo | Liga del club |
| region | Atributo | Región del club |

---

### dim_grupoedad
| Campo | Tipo | Descripción |
|------|------|------------|
| id | PK | Identificador del grupo de edad |
| grupoedad | Atributo | Rango de edad |

---

### dim_posicion
| Campo | Tipo | Descripción |
|------|------|------------|
| id | PK | Identificador de la posición |
| club_position | Atributo | Posición del jugador |

---

# ============================================================
# FACT TABLES
# ============================================================

### fact_valor_club_temporada

| Campo | Tipo | Descripción |
|------|------|------------|
| season_id | FK | Referencia a dim_season |
| club_id | FK | Referencia a dim_club |
| avg_value_eur | Medida | Promedio del valor de mercado |

**Granularidad:** Club por temporada

---

### fact_valor_region_liga

| Campo | Tipo | Descripción |
|------|------|------------|
| club_id | FK | Referencia a dim_club |
| total_value_eur | Medida | Valor total de mercado |

**Granularidad:** Club  
**Nota:** Región y liga se obtienen desde `dim_club`

---

### fact_rendimiento_edad_posicion

| Campo | Tipo | Descripción |
|------|------|------------|
| grupoedad_id | FK | Referencia a dim_grupoedad |
| posicion_id | FK | Referencia a dim_posicion |
| avg_overall | Medida | Promedio de rendimiento |

**Granularidad:** Grupo de edad + posición

---

### fact_margen_club_liga

| Campo | Tipo | Descripción |
|------|------|------------|
| club_id | FK | Referencia a dim_club |
| total_margen | Medida | Margen total |

**Granularidad:** Club

---

### fact_drilldown_mercado

| Campo | Tipo | Descripción |
|------|------|------------|
| club_id | FK | Referencia a dim_club |
| avg_value_eur | Medida | Promedio del valor de mercado |
| avg_wage_eur | Medida | Promedio del salario |

**Granularidad:** Club
**Nota:** Drill-down (club → liga → región) mediante `dim_club`

---

# ============================================================
# RELACIONES
# ============================================================

| Fact Table | Dimensiones relacionadas |
|-----------|-------------------------|
| fact_valor_club_temporada | dim_club, dim_season |
| fact_valor_region_liga |  dim_club |
| fact_rendimiento_edad_posicion | dim_grupoedad, dim_posicion |
| fact_margen_club_liga | dim_club |
| fact_drilldown_mercado | dim_club |

---

# ============================================================
# CONCLUSIÓN
# ============================================================

El modelo sigue un esquema estrella donde:

- Las dimensiones contienen atributos descriptivos
- Las fact tables contienen únicamente FKs y medidas
- La jerarquía club → liga → región se maneja en dim_club
- El análisis se realiza mediante joins entre facts y dimensiones

# DEFINICIÓN DE FACT TABLES


In [ ]:
# ============================================================
# FACT 1: fact_valor_club
# Granularidad: club por temporada
# ============================================================
fact_valor_club = df.copy()

fact_valor_club = pd.merge(
    fact_valor_club,
    dt_club_dimension[['club_name', 'club_id']],
    on='club_name',
    how='left'
)

fact_valor_club = pd.merge(
    fact_valor_club,
    dt_season_dimension[['Season', 'season_id']],
    on='Season',
    how='left'
)

fact_valor_club = fact_valor_club.groupby(
    ['club_id', 'season_id']
).agg(
    avg_value_eur=('value_eur', 'mean')
).reset_index()

display(fact_valor_club.head())


,club_id,season_id,avg_value_eur
0,1,1,1.640000e+07
1,1,2,2.093167e+07
2,1,3,2.248712e+07
3,1,4,2.978000e+07
4,1,5,2.852250e+07


In [ ]:
# ============================================================
# FACT 2: fact_valor_region_liga
# Granularidad: club por temporada (region y liga salen de dim_club)
# ============================================================
fact_valor_region_liga = df.copy()

fact_valor_region_liga = pd.merge(
    fact_valor_region_liga,
    dt_club_dimension[['club_name', 'club_id']],
    on='club_name',
    how='left'
)


fact_valor_region_liga = fact_valor_region_liga.groupby(
    ['club_id']
).agg(
    sum_value_eur=('value_eur', 'sum')
).reset_index()

display(fact_valor_region_liga.head())



,club_id,sum_value_eur
0,1,7877205000
1,2,6615990000
2,3,7418505000
3,4,5307540000
4,5,5635255000


In [ ]:

# ============================================================
# FACT 3: fact_rendimiento_posicion_edadgrupo
# Granularidad: GrupoEdad + posición + temporada
# ============================================================
fact_rendimiento_posicion_edadgrupo = df.copy()

fact_rendimiento_posicion_edadgrupo = pd.merge(
    fact_rendimiento_posicion_edadgrupo,
    dt_grupoEdad_dimension[['GrupoEdad', 'grupoEdad_id']],
    on='GrupoEdad',
    how='left'
)

fact_rendimiento_posicion_edadgrupo = pd.merge(
    fact_rendimiento_posicion_edadgrupo,
    dt_position_dimension[['club_position', 'position_id']],
    on='club_position',
    how='left'
)


fact_rendimiento_posicion_edadgrupo = fact_rendimiento_posicion_edadgrupo.groupby(
    ['grupoEdad_id', 'position_id']
).agg(
    avg_overall=('overall', 'mean')
).reset_index()

display(fact_rendimiento_posicion_edadgrupo.head())


,grupoEdad_id,position_id,avg_overall
0,1,1,70.330769
1,1,2,69.382160
2,1,3,64.896547
3,1,4,68.289183
4,1,5,67.424311


In [ ]:
# ============================================================
# FACT 4: fact_margen_club
# Granularidad: club por temporada
# ============================================================
fact_margen_club = df.copy()

if "margen" not in fact_margen_club.columns:
    fact_margen_club["margen"] = fact_margen_club["value_eur"] - fact_margen_club["wage_eur"]

fact_margen_club = pd.merge(
    fact_margen_club,
    dt_club_dimension[['club_name', 'club_id']],
    on='club_name',
    how='left'
)


fact_margen_club = fact_margen_club.groupby(
    ['club_id']
).agg(
    total_margen=('margen', 'sum')
).reset_index()

display(fact_margen_club.head())



,club_id,total_margen
0,1,7836405350
1,2,6579984000
2,3,7393062000
3,4,5288242950
4,5,5607729950


In [ ]:

# ============================================================
# FACT 5: fact_drilldown_mercado
# Granularidad: club por temporada
# ============================================================
fact_drilldown_mercado = df.copy()

fact_drilldown_mercado = pd.merge(
    fact_drilldown_mercado,
    dt_club_dimension[['club_name', 'club_id']],
    on='club_name',
    how='left'
)


fact_drilldown_mercado = fact_drilldown_mercado.groupby(
    ['club_id']
).agg(
    avg_value_eur=('value_eur', 'mean'),
    avg_wage_eur=('wage_eur', 'mean')
).reset_index()

display(fact_drilldown_mercado.head())

,club_id,avg_value_eur,avg_wage_eur
0,1,2.477108e+07,128300.786164
1,2,2.604720e+07,141755.905512
2,3,2.717401e+07,93197.802198
3,4,2.230059e+07,81080.042017
4,5,1.783309e+07,87104.588608


# Crear un small pipeline que “carga” datos temporada tras temporada.

### **EJECUTAR SOLAMENTE ESTO**

In [ ]:
!pip install sqlalchemy
!pip install pymysql
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 48.4 MB/s eta 0:00:00


In [ ]:
!apt-get update
!apt-get install -y mysql-server
!service mysql start

!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root';"
!mysql -e "FLUSH PRIVILEGES;"

print("MySQL server instalado y corriendo en Colab.")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,623 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,151 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]


In [ ]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

# ============================================================
# PARÁMETROS DE CONEXIÓN
# ============================================================
MYSQL_HOST = 'interchange.proxy.rlwy.net'
MYSQL_PORT = int('46884')
MYSQL_USER = 'root'
MYSQL_PASSWORD = 'vXYdycMCrhBbOVFijEZAWhQZIuvjGGnj'
DATABASE_NAME = 'railway'

# ============================================================
# FUNCIÓN: CREAR CONEXIÓN
# ============================================================
def create_connection(database=None):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=database,
            port=MYSQL_PORT
        )
        return conn
    except Error as e:
        print(f"Error de conexión: {e}")
        return None

In [ ]:
# ============================================================
# FUNCIÓN: GENERAR CREATE TABLE SQL
# ============================================================
def generate_create_table_sql(table_name, schema, auto_id=True):
    columns_sql_parts = []
    foreign_keys_sql_parts = []

    if auto_id:
        columns_sql_parts.append("id INT AUTO_INCREMENT PRIMARY KEY")

    for col, dtype in schema.items():
        if str(col).startswith("CONSTRAINT"):
            foreign_keys_sql_parts.append(f"{col} {dtype}")
        else:
            columns_sql_parts.append(f"`{col}` {dtype}")

    sql_parts = columns_sql_parts + foreign_keys_sql_parts
    columns_definition = ", ".join(sql_parts)

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        {columns_definition}
    )
    """
    return create_sql

## __________________ HASTA ACÁ ___________________________

In [ ]:
# ============================================================
# 1) CREAR BASE DE DATOS
# ============================================================
try:
    conn = create_connection()

    if conn is not None and conn.is_connected():
        cursor = conn.cursor()
        print(f"Base de datos '{DATABASE_NAME}' conectada correctamente.")

except Error as e:
    print(f"Error creando base de datos: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

Base de datos 'railway' conectada correctamente.


In [ ]:
# ============================================================
# 2) SCHEMA DE LA TABLA BASE
#    (dataset final limpio)
# ============================================================
players_clean_schema = {
    'season': 'VARCHAR(50)',
    'player_id': 'INT',
    'short_name': 'VARCHAR(255)',
    'age': 'INT',
    'nationality': 'VARCHAR(255)',
    'club_name': 'VARCHAR(255)',
    'league_name': 'VARCHAR(255)',
    'region': 'VARCHAR(255)',
    'player_positions': 'VARCHAR(255)',
    'club_position': 'VARCHAR(100)',
    'preferred_foot': 'VARCHAR(50)',
    'height_cm': 'INT',
    'weight_kg': 'INT',
    'overall': 'INT',
    'potential': 'INT',
    'value_eur': 'FLOAT',
    'wage_eur': 'FLOAT',
    'margen': 'FLOAT',
    'pace': 'FLOAT',
    'shooting': 'FLOAT',
    'passing': 'FLOAT',
    'dribbling': 'FLOAT',
    'defending': 'FLOAT',
    'physic': 'FLOAT',
    'posicion_principal': 'VARCHAR(100)',
    'preferred_foot_valido': 'BOOLEAN',
    'grupoaltura': 'VARCHAR(100)',
    'grupovalor': 'VARCHAR(100)',
    'grupoedad': 'VARCHAR(100)',
    'brechapotencial': 'FLOAT',
    'posicionprincipal': 'VARCHAR(100)',
    'eficienciafinanciera': 'FLOAT',
    'flag_habilidades_bajas': 'BOOLEAN',
    'flag_valor_salario': 'BOOLEAN',
    'edad_sospechosa': 'BOOLEAN'
}

In [ ]:
# ============================================================
# 3) CREAR TABLA BASE EN MYSQL
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():
        cursor = conn.cursor()
        create_table_sql = generate_create_table_sql('players_clean', players_clean_schema)
        cursor.execute(create_table_sql)
        conn.commit()
        print("Tabla 'players_clean' creada correctamente.")

except Error as e:
    print(f"Error creando tabla: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

Tabla 'players_clean' creada correctamente.


In [ ]:
# ============================================================
# FUNCIÓN: NORMALIZAR NOMBRES DE COLUMNAS
# ============================================================
def normalize_col_name(col):
    return str(col).strip().lower().replace(" ", "_")

In [ ]:
# ============================================================
# FUNCIÓN: CARGAR DATAFRAME A MYSQL
# ============================================================
def load_table(df, table_name, connection):
    cursor = connection.cursor()

    cols = "`,`".join([normalize_col_name(col) for col in df.columns])
    placeholders = ", ".join(["%s"] * df.shape[1])
    sql = f"INSERT INTO `{table_name}` (`{cols}`) VALUES ({placeholders})"

    try:
        data_tuples = [tuple(row) for row in df.values]
        cursor.executemany(sql, data_tuples)
        connection.commit()
        print(f"Se insertaron {len(data_tuples)} filas en la tabla '{table_name}'.")
    except Error as e:
        print(f"Error insertando datos en {table_name}: {e}")
        connection.rollback()
    finally:
        cursor.close()

In [ ]:
# ============================================================
# FUNCIÓN: LIMPIAR / PREPARAR COLUMNAS PARA MYSQL
def prepare_players_clean(df):
    df_mysql = df.copy()

    # Renombrar columnas existentes
    df_mysql = df_mysql.rename(columns={
        'Season': 'season',
        'GrupoAltura': 'grupoaltura',
        'GrupoValor': 'grupovalor',
        'GrupoEdad': 'grupoedad',
        'BrechaPotencial': 'brechapotencial',
        'PosicionPrincipal': 'posicionprincipal',
        'EficienciaFinanciera': 'eficienciafinanciera'
    })

    # Normalizar nombres
    df_mysql.columns = [normalize_col_name(col) for col in df_mysql.columns]


    # Asegurar todas las columnas del schema
    for col in players_clean_schema.keys():
        if col not in df_mysql.columns:
            df_mysql[col] = None

    # Quedarse solo con las columnas del schema
    df_mysql = df_mysql[list(players_clean_schema.keys())].copy()

    return df_mysql


In [ ]:
# ============================================================
# FUNCIÓN: PIPELINE DE CARGA POR TEMPORADA
# ============================================================
def pipeline_load_by_season(dt_players, database_name=DATABASE_NAME):
    seasons = sorted(dt_players['Season'].dropna().unique())

    try:
        conn = create_connection(database_name)

        if conn is None or not conn.is_connected():
            print("No se pudo establecer conexión con MySQL.")
            return

        for season in seasons:
            print(f"\nProcesando temporada: {season}")

            dt_temp = dt_players[dt_players['Season'] == season].copy()
            dt_temp = prepare_players_clean(dt_temp)

            load_table(dt_temp, 'players_clean', conn)

        print("\nCarga por temporadas completada correctamente.")

    except Error as e:
        print(f"Error en el pipeline: {e}")

    finally:
        if 'conn' in locals() and conn is not None and conn.is_connected():
            conn.close()
            print("Conexión cerrada.")


In [ ]:
# ============================================================
# 4) EJECUTAR PIPELINE
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()
        cursor.execute("TRUNCATE TABLE players_clean")
        conn.commit()
        print("players_clean limpiada correctamente.")
except Error as e:
    print(f"Error: {e}")
finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    # Added check for conn is not None to prevent AttributeError
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()
pipeline_load_by_season(df)

players_clean limpiada correctamente.

Procesando temporada: 2015
Se insertaron 16155 filas en la tabla 'players_clean'.

Procesando temporada: 2016
Se insertaron 15623 filas en la tabla 'players_clean'.

Procesando temporada: 2017
Se insertaron 17596 filas en la tabla 'players_clean'.

Procesando temporada: 2018
Se insertaron 17954 filas en la tabla 'players_clean'.

Procesando temporada: 2019
Se insertaron 18085 filas en la tabla 'players_clean'.

Procesando temporada: 2020
Se insertaron 18483 filas en la tabla 'players_clean'.

Procesando temporada: 2021
Se insertaron 18944 filas en la tabla 'players_clean'.

Procesando temporada: 2022
Se insertaron 19239 filas en la tabla 'players_clean'.

Procesando temporada: 2023
Se insertaron 56880 filas en la tabla 'players_clean'.

Procesando temporada: 2024
Se insertaron 15845 filas en la tabla 'players_clean'.

Carga por temporadas completada correctamente.
Conexión cerrada.


In [ ]:
# ============================================================
# 5) VERIFICAR DATOS CARGADOS
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():
        query = "SELECT season, COUNT(*) AS total_registros FROM players_clean GROUP BY season ORDER BY season"
        dt_verificacion = pd.read_sql(query, conn)
        display(dt_verificacion)

except Error as e:
    print(f"Error verificando datos: {e}")

finally:
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

/tmp/ipykernel_912/595374914.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dt_verificacion = pd.read_sql(query, conn)


,season,total_registros
0,2015,16155
1,2016,15623
2,2017,17596
3,2018,17954
4,2019,18085
5,2020,18483
6,2021,18944
7,2022,19239
8,2023,56880
9,2024,15845


In [ ]:
# ============================================================
# 5) CONTEO DE LOS DATOS CARGADOS
# ============================================================


import pandas as pd
import mysql.connector
from mysql.connector import Error

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():
        query = "SELECT COUNT(*) AS total_registros_global FROM players_clean"
        total_registros_global_df = pd.read_sql(query, conn)
        print("Total de registros en la tabla 'players_clean' (global):")
        display(total_registros_global_df)

except Error as e:
    print(f"Error al contar registros totales en MySQL: {e}")

finally:
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

Total de registros en la tabla 'players_clean' (global):


/tmp/ipykernel_912/3943748105.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_registros_global_df = pd.read_sql(query, conn)


,total_registros_global
0,214804


## Creación de dimensiones en MYSQL

# -------------------Correr solo en caso de hacer modificaciones a las fact tables o dimensiones

In [ ]:
# ============================================================
# LIMPIAR FACT TABLES
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

         # ----------------------------------------------------
        # 1. LIMPIAR FACT TABLES
        # ----------------------------------------------------
        cursor.execute("DELETE FROM fact_drilldown_mercado")
        cursor.execute("DELETE FROM fact_margen_club_liga")
        cursor.execute("DELETE FROM fact_rendimiento_edad_posicion")
        cursor.execute("DELETE FROM fact_valor_region_liga")
        cursor.execute("DELETE FROM fact_valor_club_temporada")

        # ----------------------------------------------------
        # 2. LIMPIAR DIMENSIONES
        # ----------------------------------------------------
        cursor.execute("DELETE FROM dim_posicion")
        cursor.execute("DELETE FROM dim_grupoedad")
        cursor.execute("DELETE FROM dim_club")
        cursor.execute("DELETE FROM dim_season")

        # ----------------------------------------------------
        # 3. REINICIAR AUTO_INCREMENT DE DIMENSIONES
        # ----------------------------------------------------
        cursor.execute("ALTER TABLE dim_season AUTO_INCREMENT = 1")
        cursor.execute("ALTER TABLE dim_club AUTO_INCREMENT = 1")
        cursor.execute("ALTER TABLE dim_grupoedad AUTO_INCREMENT = 1")
        cursor.execute("ALTER TABLE dim_posicion AUTO_INCREMENT = 1")



        conn.commit()
        print("Fact tables limpiadas correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

Error: 1146 (42S02): Table 'railway.fact_drilldown_mercado' doesn't exist


In [ ]:
# ============================================================
# FUNCION PARA CREAR VARIAS TABLAS
# ============================================================

def create_tables(tables, database_name):
    try:
        conn = create_connection(database_name)
        if conn is not None and conn.is_connected():
            cursor = conn.cursor()
            for table_name, schema in tables:
                sql = generate_create_table_sql(table_name, schema)
                cursor.execute(sql)
                print(f"Tabla '{table_name}' creada correctamente.")
            conn.commit()
    except Error as e:
        print(f"Error: {e}")
    finally:
        if 'cursor' in locals() and cursor: cursor.close()
        if 'conn' in locals() and conn.is_connected(): conn.close()


In [ ]:
# ============================================================
# Schemas para esquema ESTRELLA
# ============================================================

dim_season_schema    = {'season': 'VARCHAR(10)'}

# Club ya contiene liga y región desnormalizadas
dim_club_schema      = {
    'club_name':   'VARCHAR(255)',
    'league_name': 'VARCHAR(255)',
    'region':      'VARCHAR(255)'
}

dim_grupoedad_schema = {'grupoedad': 'VARCHAR(50)'}
dim_posicion_schema  = {'club_position': 'VARCHAR(50)'}


# 2. Crear las tablas de dimensiones
create_tables([
    ('dim_season',    dim_season_schema),
    ('dim_club',      dim_club_schema),
    ('dim_grupoedad', dim_grupoedad_schema),
    ('dim_posicion',  dim_posicion_schema),
], DATABASE_NAME)

Tabla 'dim_season' creada correctamente.
Tabla 'dim_club' creada correctamente.
Tabla 'dim_grupoedad' creada correctamente.
Tabla 'dim_posicion' creada correctamente.


## Llenar las dimensiones

In [ ]:
# ============================================================
# Dimension Season
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()


        cursor.execute("""
            INSERT INTO dim_season (season)
            SELECT DISTINCT season
            FROM players_clean
            WHERE season IS NOT NULL
        """)

        conn.commit()
        print("dim_season cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

dim_season cargada correctamente.


In [ ]:
# ============================================================
# Dimension Club
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()


        cursor.execute("""
            INSERT INTO dim_club (club_name, league_name, region)
            SELECT DISTINCT
                club_name,
                league_name,
                region
            FROM players_clean
            WHERE club_name IS NOT NULL
        """)

        conn.commit()
        print("dim_club cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

dim_club cargada correctamente.


In [ ]:
# ============================================================
# Dimension GrupoEdad
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()


        cursor.execute("""
            INSERT INTO dim_grupoedad (grupoedad)
            SELECT DISTINCT grupoedad
            FROM players_clean
            WHERE grupoedad IS NOT NULL
        """)

        conn.commit()
        print("dim_grupoedad cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

dim_grupoedad cargada correctamente.


In [ ]:
# ============================================================
# Dimension Posición
# ============================================================


try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()


        cursor.execute("""
            INSERT INTO dim_posicion (club_position)
            SELECT DISTINCT club_position
            FROM players_clean
            WHERE club_position IS NOT NULL
        """)


        conn.commit()
        print("dim_posicion cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

dim_posicion cargada correctamente.


## Creación de fact tables en MYSQL

In [ ]:
# ============================================================
# SCHEMAS FACT TABLES
# ============================================================

fact_valor_club_temporada_schema = {
    'season_id': 'INT',
    'club_id': 'INT',
    'avg_value_eur': 'FLOAT',
    'CONSTRAINT fk_fact_valor_club_temporada_season': 'FOREIGN KEY (season_id) REFERENCES dim_season(id) ON DELETE CASCADE',
    'CONSTRAINT fk_fact_valor_club_temporada_club': 'FOREIGN KEY (club_id) REFERENCES dim_club(id) ON DELETE CASCADE'
}

fact_valor_region_liga_schema = {
    'club_id': 'INT',
    'total_value_eur': 'FLOAT',
    'CONSTRAINT fk_fact_valor_region_liga_club': 'FOREIGN KEY (club_id) REFERENCES dim_club(id) ON DELETE CASCADE'
}

fact_rendimiento_edad_posicion_schema = {
    'grupoedad_id': 'INT',
    'posicion_id': 'INT',
    'avg_overall': 'FLOAT',
    'CONSTRAINT fk_fact_rendimiento_edad_posicion_grupoedad': 'FOREIGN KEY (grupoedad_id) REFERENCES dim_grupoedad(id) ON DELETE CASCADE',
    'CONSTRAINT fk_fact_rendimiento_edad_posicion_posicion': 'FOREIGN KEY (posicion_id) REFERENCES dim_posicion(id) ON DELETE CASCADE'
}

fact_margen_club_liga_schema = {
    'club_id': 'INT',
    'avg_margen': 'FLOAT',
    'total_margen': 'FLOAT',
    'CONSTRAINT fk_fact_margen_club_liga_club': 'FOREIGN KEY (club_id) REFERENCES dim_club(id) ON DELETE CASCADE'
}

fact_drilldown_mercado_schema = {
    'club_id': 'INT',
    'avg_value_eur': 'FLOAT',
    'avg_wage_eur': 'FLOAT',
    'CONSTRAINT fk_fact_drilldown_mercado_club': 'FOREIGN KEY (club_id) REFERENCES dim_club(id) ON DELETE CASCADE'
}
create_tables([
    ('fact_valor_club_temporada', fact_valor_club_temporada_schema),
    ('fact_valor_region_liga', fact_valor_region_liga_schema),
    ('fact_rendimiento_edad_posicion', fact_rendimiento_edad_posicion_schema),
    ('fact_margen_club_liga', fact_margen_club_liga_schema),
    ('fact_drilldown_mercado', fact_drilldown_mercado_schema),
], DATABASE_NAME)

Tabla 'fact_valor_club_temporada' creada correctamente.
Tabla 'fact_valor_region_liga' creada correctamente.
Tabla 'fact_rendimiento_edad_posicion' creada correctamente.
Tabla 'fact_margen_club_liga' creada correctamente.
Tabla 'fact_drilldown_mercado' creada correctamente.


In [ ]:
fact_margen_club_liga_schema = {
    'club_id': 'INT',
    'total_margen': 'FLOAT',
    'CONSTRAINT fk_fact_margen_club_liga_club': 'FOREIGN KEY (club_id) REFERENCES dim_club(id) ON DELETE CASCADE'
}
create_tables([
    ('fact_margen_club_liga', fact_margen_club_liga_schema),
], DATABASE_NAME)


Tabla 'fact_margen_club_liga' creada correctamente.


## Llenar las fact tables

In [ ]:
# ============================================================
# FACT: fact_valor_club_temporada
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        cursor.execute("TRUNCATE TABLE fact_valor_club_temporada")

        cursor.execute("""
            INSERT INTO fact_valor_club_temporada
            (season_id, club_id, avg_value_eur)
            SELECT
                ds.id AS season_id,
                dc.id AS club_id,
                AVG(pc.value_eur) AS avg_value_eur
            FROM players_clean pc
            JOIN dim_season ds
                ON pc.season = ds.season
            JOIN dim_club dc
                ON pc.club_name = dc.club_name
               AND pc.league_name = dc.league_name
               AND COALESCE(pc.region, 'Unknown') = COALESCE(dc.region, 'Unknown')
            WHERE pc.value_eur IS NOT NULL
            GROUP BY ds.id, dc.id
        """)

        conn.commit()
        print("fact_valor_club_temporada cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()


fact_valor_club_temporada cargada correctamente.


In [ ]:

# ============================================================
# FACT: fact_valor_region_liga
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        cursor.execute("TRUNCATE TABLE fact_valor_region_liga")

        cursor.execute("""
            INSERT INTO fact_valor_region_liga
            ( club_id, total_value_eur)
            SELECT
                dc.id AS club_id,
                SUM(pc.value_eur) AS total_value_eur
            FROM players_clean pc
            JOIN dim_club dc
                ON pc.club_name = dc.club_name
               AND pc.league_name = dc.league_name
               AND COALESCE(pc.region, 'Unknown') = COALESCE(dc.region, 'Unknown')
            WHERE pc.value_eur IS NOT NULL
            GROUP BY dc.id
        """)

        conn.commit()
        print("fact_valor_region_liga cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

fact_valor_region_liga cargada correctamente.


In [ ]:

# ============================================================
# FACT: fact_rendimiento_edad_posicion
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        cursor.execute("TRUNCATE TABLE fact_rendimiento_edad_posicion")

        cursor.execute("""
            INSERT INTO fact_rendimiento_edad_posicion
            (grupoedad_id, posicion_id, avg_overall)
            SELECT
                dg.id AS grupoedad_id,
                dp.id AS posicion_id,
                AVG(pc.overall) AS avg_overall
            FROM players_clean pc
            JOIN dim_grupoedad dg
                ON pc.grupoedad = dg.grupoedad
            JOIN dim_posicion dp
                ON pc.club_position = dp.club_position
            WHERE pc.overall IS NOT NULL
            GROUP BY dg.id, dp.id
        """)

        conn.commit()
        print("fact_rendimiento_edad_posicion cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

fact_rendimiento_edad_posicion cargada correctamente.


In [ ]:

# ============================================================
# FACT: fact_margen_club_liga
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        cursor.execute("TRUNCATE TABLE fact_margen_club_liga")

        cursor.execute("""
            INSERT INTO fact_margen_club_liga
            (club_id, total_margen)
            SELECT
                dc.id AS club_id,
                SUM(pc.margen) AS total_margen
            FROM players_clean pc
            JOIN dim_club dc
                ON pc.club_name = dc.club_name
               AND pc.league_name = dc.league_name
               AND COALESCE(pc.region, 'Unknown') = COALESCE(dc.region, 'Unknown')
            WHERE pc.margen IS NOT NULL
            GROUP BY dc.id
        """)

        conn.commit()
        print("fact_margen_club_liga cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()


fact_margen_club_liga cargada correctamente.


In [ ]:
# ============================================================
# FACT: fact_drilldown_mercado
# ============================================================
try:
    conn = create_connection(DATABASE_NAME)
    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        cursor.execute("TRUNCATE TABLE fact_drilldown_mercado")

        cursor.execute("""
            INSERT INTO fact_drilldown_mercado
            ( club_id, avg_value_eur, avg_wage_eur)
            SELECT
                dc.id AS club_id,
                AVG(pc.value_eur) AS avg_value_eur,
                AVG(pc.wage_eur) AS avg_wage_eur
            FROM players_clean pc
            JOIN dim_club dc
                ON pc.club_name = dc.club_name
               AND pc.league_name = dc.league_name
               AND COALESCE(pc.region, 'Unknown') = COALESCE(dc.region, 'Unknown')
            WHERE pc.value_eur IS NOT NULL
            GROUP BY dc.id
        """)

        conn.commit()
        print("fact_drilldown_mercado cargada correctamente.")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor is not None:
        cursor.close()
    if 'conn' in locals() and conn is not None and conn.is_connected():
        conn.close()

fact_drilldown_mercado cargada correctamente.


# Consultas con su respectiva interpretación:

## Consulta 1: Top 10 clubes por promedio de valor de mercado por temporada.

In [ ]:
# ============================================================
# QUERY: Top 10 clubes por valor promedio por temporada
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        query = """
        SELECT *
        FROM (
            SELECT
                ds.season,
                dc.club_name,
                f.avg_value_eur,
                ROW_NUMBER() OVER (
                    PARTITION BY ds.season
                    ORDER BY f.avg_value_eur DESC
                ) AS ranking
            FROM fact_valor_club_temporada f
            JOIN dim_season ds
                ON f.season_id = ds.id
            JOIN dim_club dc
                ON f.club_id = dc.id
        ) ranked
        WHERE ranking <= 10;
        """

        df_consulta1 = pd.read_sql(query, conn)

        display(df_consulta1)


except Error as e:
    print(f"Error: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

/tmp/ipykernel_17049/1184976749.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_consulta1 = pd.read_sql(query, conn)


,season,club_name,avg_value_eur,ranking
0,2015,FC Barcelona,16400000.0,1
1,2015,FC Bayern München,16384000.0,2
2,2015,Real Madrid CF,12557300.0,3
3,2015,Paris Saint-Germain,10688300.0,4
4,2015,Manchester City,9519540.0,5
...,...,...,...,...
95,2024,Al Nassr,52000000.0,6
96,2024,Aston Villa,51000000.0,7
97,2024,Al Hilal,50500000.0,8
98,2024,OM,39000000.0,9


### Interpretación — Consulta 1


Esta consulta identifica los 10 clubes con mayor valor de mercado promedio por jugador, segmentado por temporada, usando una ventana de ranking (ROW_NUMBER OVER PARTITION BY season).

-  Los clubes que aparecen con alta frecuencia en el top 10 a lo largo de varias temporadas son los grandes equipos europeos (Real Madrid, Barcelona, Manchester City, PSG, FC Bayern München, entre otros), lo que refleja su capacidad sostenida de atraer y retener jugadores de alto valor de mercado.
-  El valor promedio por jugador (y no el total) es una métrica más justa para comparar clubes de diferente tamaño de plantilla: un club con 25 jugadores de élite es más valioso en promedio que uno con 35 jugadores donde muchos son de valor bajo.
-  Se puede observar variación entre temporadas: clubes que ascienden al top en una temporada específica posiblemente realizaron fichajes de alto perfil ese año, mientras que descensos pueden indicar ventas de estrellas o depreciación del plantel.
-  Se observa un incremento significativo en los valores promedio a lo largo de los años, especialmente en temporadas recientes.

-  Esta consulta es útil para análisis de competitividad de mercado: los clubes del top 10 son los principales actores en el mercado de transferencias y marcan el precio de referencia para jugadores de élite.

-  Desde una perspectiva de negocio, un club fuera del top 10 que quiera ascender debería enfocarse en retener sus jugadores de mayor valor o realizar al menos un fichaje de alto impacto por temporada para elevar su promedio.

## Consulta 2: Valor total de mercado por región y por liga.

In [ ]:
# ============================================================
# QUERY: Valor total de mercado por región y por liga
# ============================================================

import time

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():

        query = """
        SELECT
            dc.region,
            dc.league_name,
            SUM(f.total_value_eur) AS valor_total_mercado
        FROM fact_valor_region_liga f
        JOIN dim_club dc
            ON f.club_id = dc.id
        GROUP BY dc.region, dc.league_name
        ORDER BY valor_total_mercado DESC;
        """

        df_consulta2 = pd.read_sql(query, conn)
        display(df_consulta2)

except Error as e:
    print(f"Error: {e}")

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()


/tmp/ipykernel_17049/3916947389.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_consulta2 = pd.read_sql(query, conn)


,region,league_name,valor_total_mercado
0,Europa,English Premier League,4.852289e+10
1,Europa,Spain Primera Division,4.246379e+10
2,Europa,Italian Serie A,3.148371e+10
3,Europa,German 1. Bundesliga,3.052535e+10
4,Europa,French Ligue 1,2.269657e+10
...,...,...,...
96,Europa,Veikkausliiga,5.120750e+07
97,Europa,Cypriot First Division,3.086500e+07
98,Europa,National League,2.309000e+07
99,Europa,English National League,1.101000e+07


### Interpretación — Consulta 2

Esta consulta agrupa el valor total de mercado por región geográfica y por liga, permitiendo identificar cuáles mercados concentran mayor riqueza futbolística.

- Las ligas europeas (Premier League, Primera Division, Bundesliga, Serie A, Ligue 1) dominan claramente el valor total de mercado, reflejando la mayor inversión y competitividad de dichas competiciones.
- Dentro de Europa, la Premier League se posiciona como la liga con mayor valor acumulado, seguida de Primera Division y la Italian Serie A, lo que concuerda con los presupuestos de transferencias históricos de esas ligas.
- Las regiones de América del Sur y resto del mundo presentan valores considerablemente más bajos, lo que indica que los jugadores de alto valor tienden a concentrarse en clubes europeos.
- Esta información es útil para decisiones de inversión y scouting: ligas con menor valor total pero alto potencial representan oportunidades de compra con menor costo.


## Consulta 3: Rendimiento (Overall) medio por GrupoEdad y por posición.

In [ ]:
# ============================================================
# QUERY: Rendimiento medio por GrupoEdad y posición
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():

        query = """
          SELECT
            dg.grupoedad            AS GrupoEdad,
            dp.club_position        AS Posicion,
            ROUND(f.avg_overall, 2) AS rendimiento_promedio
        FROM fact_rendimiento_edad_posicion f
        JOIN dim_grupoedad dg ON f.grupoedad_id = dg.id
        JOIN dim_posicion  dp ON f.posicion_id  = dp.id
        ORDER BY GrupoEdad, rendimiento_promedio DESC;
        """

        df_consulta3 = pd.read_sql(query, conn)
        display(df_consulta3)

except Error as e:
    print(f"Error: {e}")

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()


/tmp/ipykernel_17049/1805843512.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_consulta3 = pd.read_sql(query, conn)


,GrupoEdad,Posicion,rendimiento_promedio
0,Prime,CF,70.33
1,Prime,RAM,70.22
2,Prime,RF,69.55
3,Prime,LAM,69.41
4,Prime,LW,69.38
...,...,...,...
85,Young,CM,60.33
86,Young,CB,60.27
87,Young,GK,60.06
88,Young,SUB,58.54


### Interpretación — Consulta 3

Esta consulta muestra el overall promedio segmentado por grupo de edad y posición de juego.

-  Los jugadores en etapa veteran (>28 años) presentan los overalls más altos en la mayoría de las posiciones, destacando especialmente en posiciones ofensivas como CF, donde alcanzan valores superiores a 75. Esto sugiere que la experiencia acumulada tiene un impacto importante en el rendimiento promedio.
Por ejemplo, en la posición de delantero centro (CF), los jugadores veteran alcanzan un promedio de 75.01, mientras que los prime se sitúan alrededor de 70.33 y los jóvenes en 64.55. Este mismo patrón se repite de forma consistente en la mayoría de las posiciones analizadas.

-  Los jugadores en etapa prime (21–28 años) muestran overalls competitivos pero ligeramente menores que los veteranos, manteniéndose como un grupo de alto rendimiento, aunque sin alcanzar los valores máximos observados.
-  Los jugadores jóvenes (<21 años) presentan los overalls más bajos en prácticamente todas las posiciones, lo cual es consistente con su etapa de desarrollo y menor experiencia en el ámbito profesional.
-  Este comportamiento sugiere que, dentro de este conjunto de datos, la experiencia tiene un peso mayor en el rendimiento promedio que la edad “prime”, posiblemente porque los jugadores veteranos que permanecen activos son aquellos con mayor calidad y consistencia.
Esta segmentación es útil para estrategias de contratación: si se busca rendimiento inmediato y consolidado, los jugadores veteranos representan una opción más sólida para rendimiento inmediato; mientras que, si el objetivo es proyección a largo plazo, los jugadores jóvenes siguen siendo la opción adecuada.


## Consulta 4: Análisis de márgenes (Margen = ValorMercado - Salario) por Club y por Liga.

In [ ]:
# ============================================================
# QUERY: Análisis de márgenes por Club y por Liga
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():

        query = """
        SELECT
            dc.league_name              AS Liga,
            dc.club_name                AS Club,
            ROUND(f.total_margen, 2)    AS margen_total
        FROM fact_margen_club_liga f
        JOIN dim_club dc ON f.club_id = dc.id
        ORDER BY margen_total DESC
        LIMIT 30;
        """

        df_consulta4 = pd.read_sql(query, conn)
        display(df_consulta4)

except Error as e:
    print(f"Error: {e}")

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()


/tmp/ipykernel_17049/3813090859.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_consulta4 = pd.read_sql(query, conn)


,Liga,Club,margen_total
0,Spain Primera Division,Real Madrid CF,6.579984e+09
1,Spain Primera Division,FC Barcelona,6.134007e+09
2,English Premier League,Manchester City,5.867526e+09
3,German 1. Bundesliga,FC Bayern München,5.761846e+09
4,French Ligue 1,Paris Saint-Germain,5.288243e+09
5,English Premier League,Chelsea,4.957548e+09
6,English Premier League,Liverpool,4.944203e+09
7,English Premier League,Manchester United,4.818000e+09
8,Italian Serie A,Juventus,4.661184e+09
9,Spain Primera Division,Atlético de Madrid,4.612024e+09


### Interpretación — Consulta 4

El margen (ValorMercado − Salario) es un indicador de eficiencia financiera: un margen positivo alto indica que el club posee jugadores cuyo valor de mercado supera ampliamente su coste salarial, lo que representa un activo financiero favorable.

-  Los clubes con mayor margen total son generalmente los grandes equipos con plantillas de alto valor de mercado (p. ej., Real Madrid, Barcelona, Manchester City, PSG), lo que refleja su capacidad de generar valor a partir de sus activos deportivos.
-  Se observa una fuerte concentración del margen total en un grupo reducido de clubes, evidenciando una diferencia significativa respecto al resto.
Clubes con alto margen total pueden representar casos donde el valor de mercado de los jugadores crece más rápido que sus salarios, ya sea por desarrollo deportivo o fichajes exitosos.
-  En caso de existir márgenes negativos (no observados en este resultado), estos podrían indicar que el club está pagando más de lo que el mercado estima que valen sus jugadores, lo que podría ser una señal de ineficiencia financiera.
-  A nivel de ligas, los mayores márgenes se concentran en las principales ligas europeas, lo que confirma su dominio económico en el fútbol profesional.

## Consulta 5: Drill-down por Club → Liga → Región, con promedio de ValorMercado y Salario.

In [ ]:
# ============================================================
# QUERY: Drill-down Club → Liga → Región
# ============================================================

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():

        query = """
        SELECT
            dc.region                       AS Region,
            dc.league_name                  AS Liga,
            dc.club_name                    AS Club,
            ROUND(f.avg_value_eur, 2)       AS valor_mercado_promedio,
            ROUND(f.avg_wage_eur,  2)       AS salario_promedio
        FROM fact_drilldown_mercado f
        JOIN dim_club dc ON f.club_id = dc.id
        ORDER BY valor_mercado_promedio DESC;
        """

        df_consulta5 = pd.read_sql(query, conn)
        display(df_consulta5)

        # ── Nivel Región (roll-up) ──────────────────────────────
        print("\n--- Roll-up a nivel Región ---")
        df_region = df_consulta5.groupby('Region').agg(
            valor_mercado_promedio=('valor_mercado_promedio', 'mean'),
            salario_promedio=('salario_promedio', 'mean')
        ).round(2).reset_index().sort_values('valor_mercado_promedio', ascending=False)
        display(df_region)

        # ── Nivel Liga (roll-up intermedio) ────────────────────
        print("\n--- Roll-up a nivel Liga ---")
        df_liga = df_consulta5.groupby(['Region','Liga']).agg(
            valor_mercado_promedio=('valor_mercado_promedio', 'mean'),
            salario_promedio=('salario_promedio', 'mean')
        ).round(2).reset_index().sort_values('valor_mercado_promedio', ascending=False)
        display(df_liga)





except Error as e:
    print(f"Error: {e}")

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()


/tmp/ipykernel_17049/1503631610.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_consulta5 = pd.read_sql(query, conn)


,Region,Liga,Club,valor_mercado_promedio,salario_promedio
0,Europa,Ligue 1,Al Hilal,109000000.0,270000.0
1,Europa,Ligue 1,Inter Miami CF,100500000.0,560000.0
2,Europa,Premier League,FC Bayern München,83000000.0,200000.0
3,Europa,Premier League,Al Ittihad,52750000.0,157500.0
4,Europa,Bundesliga,Al Nassr,52000000.0,170000.0
...,...,...,...,...,...
5518,Europa,Ligue 2,Nîmes Olympique,30000.0,2000.0
5519,Europa,League One,Preston North End,25000.0,2000.0
5520,Europa,Pro League,Al Orobah,25000.0,2000.0
5521,Asia,K League 1,Jeonbuk Hyundai Motors,20000.0,2000.0



--- Roll-up a nivel Región ---


,Region,valor_mercado_promedio,salario_promedio
5,Europa,2334161.64,8966.83
1,America del Norte,1589284.98,6723.73
2,America del Sur,1036975.50,3287.76
6,Oceania,834962.62,2938.13
4,Desconocido,786800.93,3780.22
3,Asia,740704.27,3359.93
0,Africa,250861.97,1169.53



--- Roll-up a nivel Liga ---


,Region,Liga,valor_mercado_promedio,salario_promedio
53,Europa,English Premier League,7374668.39,42417.98
90,Europa,Spain Primera Division,6749828.34,28719.31
68,Europa,La Liga,6675285.49,22631.66
98,Europa,Ukrainian Premier League,5834184.50,9037.94
80,Europa,Premier League,5832157.46,22963.58
...,...,...,...,...
51,Europa,English League Two,302911.57,2682.58
25,Asia,Indian Super League,285610.14,611.68
86,Europa,Scottish Championship,267142.84,6321.43
0,Africa,Premier Division,196537.58,861.55


### Interpretación — Consulta 5 (Drill-down)

El drill-down permite analizar los datos en distintos niveles de granularidad: desde el más agregado (Región) hasta el más detallado (Club), pasando por Liga.

- A nivel Región: Europa concentra el mayor valor de mercado promedio y salarios más altos por jugador, confirmando su posición dominante en el fútbol mundial. El resto de regiones (América del Norte, América del Sur, Asia, etc.) presentan promedios notablemente más bajos.
- A nivel Liga: dentro de Europa, English Premier League y Spain Primera Division lideran en valor de mercado promedio por jugador, seguidas de la La Liga y la Ukrainian Premier League. Esto es coherente con los salarios y fichajes observados en el mercado real.
- A nivel Club: los clubes más ricos concentran los valores más altos. Equipos como Al Hilal, Inter Miami CF o FC Bayern München, destacan sobre el resto.
- El drill-down es una herramienta OLAP clave para la toma de decisiones: permite a un director deportivo comenzar evaluando regiones de interés y profundizar hasta el club específico que mejor cumple sus criterios de inversión o scouting.


## Medir tiempo de ejecución y técnicas de optimización

In [ ]:
# ============================================================
# TAREA 8: Tiempo de ejecución y optimización
# Consulta de referencia: Valor total de mercado por región y por liga
# ============================================================

import time

query_base = """
SELECT
    dc.region,
    dc.league_name,
    SUM(f.total_value_eur) AS valor_total_mercado
FROM fact_valor_region_liga f
JOIN dim_club dc ON f.club_id = dc.id
GROUP BY dc.region, dc.league_name
ORDER BY dc.region, valor_total_mercado DESC;
"""

# ── Versión SIN optimización (sin índice) ─────────────────
print("=" * 60)
print("VERSIÓN BASE (sin índice adicional)")
print("=" * 60)

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        # Eliminar índice si ya existía de una corrida anterior
        try:
            cursor.execute("DROP INDEX idx_club_id ON fact_valor_region_liga;")
            conn.commit()
            print("Índice previo eliminado.")
        except:
            print("No había índice previo, continuando.")

        tiempos_base = []
        for i in range(3):
            t0 = time.perf_counter()
            df_base = pd.read_sql(query_base, conn)
            t1 = time.perf_counter()
            tiempos_base.append(t1 - t0)

        promedio_base = sum(tiempos_base) / len(tiempos_base)
        print(f"Tiempos (3 runs): {[round(t*1000, 2) for t in tiempos_base]} ms")
        print(f"Promedio base:    {round(promedio_base*1000, 2)} ms")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()


# ── Versión CON índice en club_id ──────────────────────────
print("\n" + "=" * 60)
print("VERSIÓN OPTIMIZADA (con índice en club_id)")
print("=" * 60)

try:
    conn = create_connection(DATABASE_NAME)

    if conn is not None and conn.is_connected():
        cursor = conn.cursor()

        # MySQL no soporta CREATE INDEX IF NOT EXISTS
        # Se verifica manualmente si el índice ya existe antes de crearlo
        cursor.execute("""
            SELECT COUNT(1) AS existe
            FROM information_schema.statistics
            WHERE table_schema = %s
              AND table_name   = 'fact_valor_region_liga'
              AND index_name   = 'idx_club_id';
        """, (DATABASE_NAME,))
        existe = cursor.fetchone()[0]

        if not existe:
            cursor.execute("CREATE INDEX idx_club_id ON fact_valor_region_liga(club_id);")
            conn.commit()
            print("Índice idx_club_id creado.")
        else:
            print("Índice idx_club_id ya existía, reutilizando.")

        tiempos_opt = []
        for i in range(3):
            t0 = time.perf_counter()
            df_opt = pd.read_sql(query_base, conn)
            t1 = time.perf_counter()
            tiempos_opt.append(t1 - t0)

        promedio_opt = sum(tiempos_opt) / len(tiempos_opt)
        print(f"Tiempos (3 runs): {[round(t*1000, 2) for t in tiempos_opt]} ms")
        print(f"Promedio opt:     {round(promedio_opt*1000, 2)} ms")

        if promedio_base > 0:
            mejora = ((promedio_base - promedio_opt) / promedio_base) * 100
            print(f"\nMejora de rendimiento: {round(mejora, 1)}%")

except Error as e:
    print(f"Error: {e}")

finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()

VERSIÓN BASE (sin índice adicional)
No había índice previo, continuando.


/tmp/ipykernel_18069/1645850468.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_base = pd.read_sql(query_base, conn)


Tiempos (3 runs): [556.09, 555.49, 556.34] ms
Promedio base:    555.97 ms

VERSIÓN OPTIMIZADA (con índice en club_id)
Índice idx_club_id ya existía, reutilizando.


/tmp/ipykernel_18069/1645850468.py:89: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_opt = pd.read_sql(query_base, conn)


Tiempos (3 runs): [476.96, 478.55, 476.69] ms
Promedio opt:     477.4 ms

Mejora de rendimiento: 14.1%


### Tiempo de ejecución y optimización

Técnica aplicada: índice en clave foránea (FK)

La consulta de valor total de mercado por región y liga realiza un `JOIN` entre `fact_valor_region_liga` y `dim_club` usando `club_id`. Sin un índice sobre esa columna, MySQL realiza un full table scan en cada ejecución, recorriendo toda la tabla de hechos.

Al crear un índice `idx_club_id` sobre `fact_valor_region_liga(club_id)`, el motor puede localizar los registros relevantes directamente, reduciendo el costo de la operación de O(n) a O(log n).

**Otras técnicas de optimización aplicables en este contexto:**

| Técnica | Descripción | Cuándo aplica |
|---|---|---|
| Índices en FK | Acelera los JOINs entre fact y dim tables | Siempre en claves foráneas |
| Particionamiento por temporada | Divide la tabla de hechos por `season_id` | Cuando la consulta filtra por temporada |
| Vistas materializadas | Pre-calcula agregaciones costosas | Consultas de roll-up frecuentes |
| Columnar storage | Almacenamiento orientado a columnas | Agregaciones en datasets grandes |
| Query caching | Reutiliza resultados de queries idénticas | Dashboards con consultas repetidas |

En este dataset el impacto puede ser modesto por el tamaño de datos, pero la técnica escala significativamente en entornos productivos con millones de registros por temporada.
